<a href="https://colab.research.google.com/github/antoniopioricciardi/sam3_demo/blob/main/sam3_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers accelerate torch torchvision
!pip install kernels # used by SAM3 for better mask quality

In [8]:
!pip install -U git+https://github.com/huggingface/transformers.git

# Install this to solve AttributeError: 'CLIPTextModelOutput' object has no attribute 'shape'
# in videoiterator

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-d_sre23c
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-d_sre23c
  Resolved https://github.com/huggingface/transformers.git to commit 138c1c813c6586d5702f9719e00114c1f8e99070
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.3.0.dev0-py3-none-any.whl size=11300857 sha256=b38b929c5f872c4cb0736609db5f72148ba598604ab4853b6598fd9c7b45ca5c
  Stored in directory: /tmp/pip-ephem-wheel-cache-318aifff/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install av

ERROR: Operation cancelled by user
^C


In [1]:
from google.colab import userdata
from huggingface_hub import login

# Retrieve the token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Login to Hugging Face
if hf_token:
    login(token=hf_token, add_to_git_credential=True)
    print('Logged in successfully!')
else:
    print('Please add your HF_TOKEN to the Colab Secrets (🔑 icon) first.')

Logged in successfully!


In [2]:
from PIL import Image
import requests
from google.colab import userdata

from transformers import Sam3VideoModel, Sam3VideoProcessor
from transformers.video_utils import load_video
from accelerate import Accelerator
import torch


In [3]:
device = Accelerator().device
model = Sam3VideoModel.from_pretrained("facebook/sam3").to(device, dtype=torch.bfloat16)
processor = Sam3VideoProcessor.from_pretrained("facebook/sam3")

Loading weights:   0%|          | 0/1797 [00:00<?, ?it/s]

In [4]:
# Load video
video_url = "https://huggingface.co/datasets/hf-internal-testing/sam2-fixtures/resolve/main/bedroom.mp4"
video_frames, _ = load_video(video_url)

# Initialize session — key memory trick: keep video & processing on CPU
inference_session = processor.init_video_session(
    video=video_frames,
    inference_device=device,
    processing_device="cpu",        # preprocess on CPU to save VRAM
    video_storage_device="cpu",     # store frames on CPU
    dtype=torch.bfloat16,
)

# Add ALL text prompts at once — the model reuses vision features internally
prompts = ["person", "bed", "lamp"]
processor.add_text_prompt(inference_session, prompts)

# Single propagation pass handles all prompts
outputs_per_frame = {}
for model_outputs in model.propagate_in_video_iterator(
    inference_session=inference_session,
    max_frame_num_to_track=50,      # limit frames to save compute
    show_progress_bar=True,
):
    processed = processor.postprocess_outputs(inference_session, model_outputs)
    outputs_per_frame[model_outputs.frame_idx] = processed

# Results include a prompt → object ID mapping
frame_0 = outputs_per_frame[0]
for prompt, obj_ids in frame_0["prompt_to_obj_ids"].items():
    print(f"{prompt}: {len(obj_ids)} objects")

# frame_0["masks"]  — (num_objects, H, W) binary masks
# frame_0["boxes"]  — (num_objects, 4) XYXY absolute coords
# frame_0["scores"] — (num_objects,) confidence scores

kernels library is not installed. NMS post-processing, hole filling, and sprinkle removal will be skipped. Install it with `pip install kernels` for better mask quality.


person: 2 objects
bed: 1 objects
lamp: 1 objects


In [5]:
import numpy as np
import matplotlib

def overlay_masks(image, masks):
    image = image.convert("RGBA")
    masks = 255 * masks.cpu().numpy().astype(np.uint8)

    n_masks = masks.shape[0]
    cmap = matplotlib.colormaps.get_cmap("rainbow").resampled(n_masks)
    colors = [
        tuple(int(c * 255) for c in cmap(i)[:3])
        for i in range(n_masks)
    ]

    for mask, color in zip(masks, colors):
        mask = Image.fromarray(mask)
        overlay = Image.new("RGBA", image.size, color + (0,))
        alpha = mask.point(lambda v: int(v * 0.5))
        overlay.putalpha(alpha)
        image = Image.alpha_composite(image, overlay)
    return image

In [7]:
from PIL import Image

for frame_idx, frame in enumerate(video_frames[:50]):
    frame_output = outputs_per_frame.get(frame_idx)
    if frame_output is not None and len(frame_output["masks"]) > 0:
        # Convert numpy array frame to PIL Image before processing
        pil_frame = Image.fromarray(frame.astype('uint8'))
        overlay = overlay_masks(pil_frame, frame_output["masks"])
        overlay.save(f"frame_{frame_idx:04d}.png")